# ASOS A/B Testing Exploration

This notebook walks through the ASOS Digital Experiments Dataset to understand how AB testing behaves across conversion, revenue, and engagement metrics. The goal is to compute lift, confidence, and statistical significance for representative challenger variants and then save the visual artifacts for future storytelling.

In [1]:
import math
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns

sns.set(style='whitegrid', palette='muted')

BASE_DIR = Path('..')
DATA_PATH = BASE_DIR / 'data' / 'asos_digital_experiments_dataset.csv'
OUTPUTS_DIR = BASE_DIR / 'outputs'
PLOTS_DIR = OUTPUTS_DIR / 'plots'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

,experiment_id,variant_id,metric_id,time_since_start,count_c,count_t,mean_c,mean_t,variance_c,variance_t
0,036afc,2,1,1.5,188065.0,186686.0,0.107808,0.107828,0.096186,0.096201
1,036afc,2,1,2.0,245041.0,243694.0,0.131790,0.131435,0.114422,0.114160
2,036afc,2,1,2.5,277237.0,275949.0,0.143065,0.142711,0.122598,0.122345
3,036afc,2,1,3.0,315689.0,314676.0,0.161789,0.160997,0.135613,0.135077
4,036afc,2,1,3.5,338631.0,337715.0,0.172474,0.171067,0.142727,0.141803


## Dataset overview

We read all observed variants, metrics, and cumulative statistics from the CSV. The first cell above shows the raw columns. The code below summarizes how many experiments and metrics are present to orient the analysis.

In [2]:
overview = pd.DataFrame({
    'distinct_experiments': [df['experiment_id'].nunique()],
    'distinct_metrics': [df['metric_id'].nunique()],
    'total_rows': [len(df)]
})

duration = df.groupby('experiment_id')['time_since_start'].max().describe()
print('Duration (in hours) summary for experiments:')
print(duration.round(1))

display(overview.T)

Duration (in hours) summary for experiments:
count     78.0
mean      46.6
std       25.1
min        2.0
25%       32.0
50%       43.5
75%       57.4
max      131.0
Name: time_since_start, dtype: float64


,0
distinct_experiments,78
distinct_metrics,4
total_rows,24153


## From control to challenger variants

Rather than join across variant IDs, we take advantage of the fact that each row already stores control (`mean_c`, `count_c`, `variance_c`) and treatment (`mean_t`, `count_t`, `variance_t`) absolute statistics for that challenger variant. We just need the latest snapshot per experiment, metric, and variant combination.

In [3]:
analysis_df = (
    df[df['variant_id'] != 1]
    .sort_values('time_since_start')
    .groupby(['experiment_id', 'metric_id', 'variant_id'], as_index=False)
    .last()
)
summary = analysis_df.rename(columns={
    'count_c': 'count_control',
    'count_t': 'count_treatment',
    'mean_c': 'mean_control',
    'mean_t': 'mean_treatment',
    'variance_c': 'variance_control',
    'variance_t': 'variance_treatment',
})

summary['lift'] = summary['mean_treatment'] - summary['mean_control']
summary['relative_lift'] = summary['lift'] / summary['mean_control'].replace(0, np.nan)
summary['std_error'] = np.sqrt(
    summary['variance_control'] / summary['count_control'] +
    summary['variance_treatment'] / summary['count_treatment']
)
summary['z_score'] = summary['lift'] / summary['std_error']

summary['p_value'] = summary['z_score'].map(lambda z: 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2)))))
summary['significant'] = summary['p_value'] < 0.05
summary = summary.replace([np.inf, -np.inf], np.nan).dropna(subset=['std_error'])

summary.head(8)

,experiment_id,metric_id,variant_id,time_since_start,count_control,count_treatment,mean_control,mean_treatment,variance_control,variance_treatment,lift,relative_lift,std_error,z_score,p_value,significant
0,036afc,1,2,76.5,1050994.0,1050508.0,0.586768,0.587275,0.242471,0.242383,0.000507,0.000863,0.000679,0.745753,0.455817,False
1,036afc,2,2,76.5,1050994.0,1050508.0,2.123119,2.116594,11.830135,11.710985,-0.006525,-0.003073,0.004733,-1.378520,0.168043,False
2,036afc,3,2,76.5,1050994.0,1050508.0,5.966040,5.959434,109.673591,109.676649,-0.006606,-0.001107,0.014448,-0.457235,0.647502,False
3,036afc,4,2,76.5,1050994.0,1050508.0,148.194364,147.904168,78105.387371,77913.811421,-0.290197,-0.001958,0.385336,-0.753101,0.451389,False
4,329386,1,2,91.0,3892143.0,3774952.0,0.220356,0.220401,0.171799,0.171824,0.000045,0.000203,0.000299,0.149309,0.881310,False
5,329386,2,2,91.0,3892143.0,3774952.0,0.484668,0.485993,1.940506,1.973676,0.001325,0.002734,0.001011,1.311070,0.189834,False
6,329386,3,2,91.0,3892143.0,3774952.0,1.514490,1.516622,24.628636,24.909454,0.002131,0.001407,0.003595,0.592759,0.553342,False
7,329386,4,2,91.0,3892143.0,3774952.0,34.595619,34.523872,14733.585553,14517.510856,-0.071747,-0.002074,0.087357,-0.821308,0.411471,False


## Capture final summary

Save the filtered summary and highlight significant lifts. This table is what the Streamlit app and any downstream storytelling will reuse.

In [4]:
summary_path = OUTPUTS_DIR / 'experiment_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved a cleaned summary to', summary_path.name)

Saved a cleaned summary to experiment_summary.csv


## Visual storytelling: lift by experiment

This grouped view surfaces which experiments produced the largest absolute lift so you can immediately recognize standout results.

In [5]:
largest_lifts = (
    summary.assign(abs_lift=lambda df: df['lift'].abs())
    .sort_values('abs_lift', ascending=False)
    .head(10)
    .assign(label=lambda df: df['experiment_id'] + ' (metric ' + df['metric_id'].astype(str) + ')')
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=largest_lifts,
    x='lift',
    y='label',
    hue='significant',
    dodge=False,
    palette=['#4c78a8', '#f58518'],
)
plt.axvline(0, color='gray', linewidth=0.8)
plt.title('Top absolute lift across challenger variants')
plt.xlabel('Mean difference (treatment - control)')
plt.ylabel('Experiment (metric)')
plt.legend(title='Statistically significant')
plt.gca().xaxis.set_major_formatter(ticker.StrMethodFormatter('{x:.3f}'))
plt.tight_layout()
plot_path = PLOTS_DIR / 'metric_lift_bar.png'
plt.savefig(plot_path)
plt.close()
print('Saved lift bar chart to', plot_path.name)

Saved lift bar chart to metric_lift_bar.png


## Evolution over time for a representative experiment

Choose the experiment with the most data points and plot both variants for the metric that has the longest run. This visual demonstrates how the treatment curve diverges from the control as the experiment progresses.

In [6]:
selected_experiment = df['experiment_id'].value_counts().idxmax()
exp_df = df[df['experiment_id'] == selected_experiment]
metric_of_interest = exp_df['metric_id'].mode().iloc[0]
exp_metric = exp_df[exp_df['metric_id'] == metric_of_interest].sort_values('time_since_start')

plt.figure(figsize=(10, 5))
for variant in sorted(exp_metric['variant_id'].unique()):
    variant_df = exp_metric[exp_metric['variant_id'] == variant]
    plt.plot(
        variant_df['time_since_start'],
        variant_df['mean_t'],
        label=f'Variant {variant} (mean treatment)',
        linestyle='-' if variant == 2 else '--',
    )
    plt.plot(
        variant_df['time_since_start'],
        variant_df['mean_c'],
        label=f'Variant {variant} (mean control)',
        linestyle=':' if variant == 2 else '-.',
    )
plt.title(f'Experiment {selected_experiment} · Metric {metric_of_interest} over time')
plt.xlabel('Time since experiment start (hours)')
plt.ylabel('Reported metric')
plt.legend(loc='best', fontsize='small')
plt.tight_layout()
time_plot_path = PLOTS_DIR / 'time_series_selected_experiment.png'
plt.savefig(time_plot_path)
plt.close()
print('Saved time-series chart to', time_plot_path.name)

Saved time-series chart to time_series_selected_experiment.png
